In [1]:
!git clone https://github.com/tamari1990/facial-expression-recognition.git

Cloning into 'facial-expression-recognition'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [2]:
%cd facial-expression-recognition

/content/facial-expression-recognition


In [3]:
!pip install kaggle wandb -q

In [5]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('challenges-in-representation-learning-facial-expression-recognition-challenge')

print("Path to competition files:", path)

100%|██████████| 285M/285M [00:02<00:00, 114MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge


In [6]:
import os

path = "/root/.cache/kagglehub/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge"

for f in os.listdir(path):
    print(f)

example_submission.csv
fer2013.tar.gz
icml_face_data.csv
test.csv
train.csv


In [7]:
import pandas as pd

df = pd.read_csv(f"{path}/train.csv")
print(df.shape)
print(df.head())
print(df['emotion'].value_counts())

(28709, 2)
   emotion                                             pixels
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1        0  151 150 147 155 148 133 111 140 170 174 182 15...
2        2  231 212 156 164 174 138 161 173 182 200 106 38...
3        4  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4        6  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...
emotion
3    7215
6    4965
4    4830
2    4097
0    3995
5    3171
1     436
Name: count, dtype: int64


In [8]:
!mkdir -p /content/facial-expression-recognition/src

In [9]:
%%writefile /content/facial-expression-recognition/src/dataset.py

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class FERDataset(Dataset):
    emotion_labels = {
        0: 'Angry',
        1: 'Disgust',
        2: 'Fear',
        3: 'Happy',
        4: 'Sad',
        5: 'Surprise',
        6: 'Neutral'
    }

    def __init__(self, csv_path, transform=None):
        df = pd.read_csv(csv_path)
        self.emotions = df['emotion'].values
        self.pixels = df['pixels'].values
        self.transform = transform

    def __len__(self):
        return len(self.emotions)

    def __getitem__(self, idx):
        image = np.array(self.pixels[idx].split(), dtype=np.float32)
        image = image.reshape(48, 48)

        image = image / 255.0

        image = torch.tensor(image).unsqueeze(0)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.emotions[idx], dtype=torch.long)
        return image, label


def get_dataloaders(csv_path, batch_size=64, val_split=0.2):
    from torch.utils.data import random_split

    dataset = FERDataset(csv_path)

    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

Writing /content/facial-expression-recognition/src/dataset.py


In [10]:
import sys
sys.path.append('/content/facial-expression-recognition')

from src.dataset import FERDataset, get_dataloaders

train_loader, val_loader = get_dataloaders(f"{path}/train.csv")

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

Batch shape: torch.Size([64, 1, 48, 48])
Labels shape: torch.Size([64])


In [11]:
%%writefile /content/facial-expression-recognition/src/models.py

import torch
import torch.nn as nn

Writing /content/facial-expression-recognition/src/models.py


In [12]:
%%writefile -a /content/facial-expression-recognition/src/models.py

# ================================
# Model 1: Simple CNN (baseline)
# ================================
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # 48 → 24

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)               # 24 → 12
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

Appending to /content/facial-expression-recognition/src/models.py


In [13]:
%%writefile -a /content/facial-expression-recognition/src/models.py

# ================================
# Model 2: Medium CNN
# ================================
class MediumCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(MediumCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # 48 → 24

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # 24 → 12

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # 12 → 6

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)               # 6 → 3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

Appending to /content/facial-expression-recognition/src/models.py


In [14]:
%%writefile -a /content/facial-expression-recognition/src/models.py

class DeepCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.25):
        super(DeepCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 1024),
            nn.ReLU(),
            nn.Dropout(dropout * 2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout * 2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

Appending to /content/facial-expression-recognition/src/models.py


In [15]:
%%writefile /content/facial-expression-recognition/src/models.py

import torch
import torch.nn as nn

# Model 1: Simple CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Model 2: Medium CNN
class MediumCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(MediumCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Model 3: Deep CNN
class DeepCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.25):
        super(DeepCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 1024),
            nn.ReLU(),
            nn.Dropout(dropout * 2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout * 2),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

Overwriting /content/facial-expression-recognition/src/models.py


In [16]:
import torch
import sys
sys.path.append('/content/facial-expression-recognition')

from src.models import SimpleCNN, MediumCNN, DeepCNN

x = torch.randn(4, 1, 48, 48)

for Model, name in [(SimpleCNN, "SimpleCNN"), (MediumCNN, "MediumCNN"), (DeepCNN, "DeepCNN")]:
    model = Model()
    out = model(x)
    params = sum(p.numel() for p in model.parameters())
    print(f"{name}: output={out.shape}, params={params:,}")

SimpleCNN: output=torch.Size([4, 7]), params=595,655
MediumCNN: output=torch.Size([4, 7]), params=1,572,551
DeepCNN: output=torch.Size([4, 7]), params=11,112,647


In [17]:
%%writefile /content/facial-expression-recognition/src/train.py

import torch
import torch.nn as nn
import wandb
from tqdm import tqdm

Writing /content/facial-expression-recognition/src/train.py


In [18]:
%%writefile -a /content/facial-expression-recognition/src/train.py

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total

Appending to /content/facial-expression-recognition/src/train.py


In [19]:
%%writefile -a /content/facial-expression-recognition/src/train.py

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

Appending to /content/facial-expression-recognition/src/train.py


In [20]:
%%writefile -a /content/facial-expression-recognition/src/train.py

def train(model, train_loader, val_loader, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    # WandB run
    wandb.init(
        project="facial-expression-recognition",
        name=config["run_name"],
        config=config
    )
    wandb.watch(model, log="all")

    best_val_acc = 0

    for epoch in range(config["epochs"]):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = val_epoch(model, val_loader, criterion, device)

        # WandB ლოგი
        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
        })

        print(f"Epoch {epoch+1}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{config['run_name']}_best.pth")

    wandb.finish()
    print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Appending to /content/facial-expression-recognition/src/train.py


In [21]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tgela23 (tgela23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [27]:
x = torch.randn(4, 1, 48, 48)
for Model, name in [(SimpleCNN, "SimpleCNN"), (MediumCNN, "MediumCNN"), (DeepCNN, "DeepCNN")]:
    model = Model()
    out = model(x)
    print(f"{name}: input={x.shape} → output={out.shape}")

SimpleCNN: input=torch.Size([4, 1, 48, 48]) → output=torch.Size([4, 7])
MediumCNN: input=torch.Size([4, 1, 48, 48]) → output=torch.Size([4, 7])
DeepCNN: input=torch.Size([4, 1, 48, 48]) → output=torch.Size([4, 7])


In [28]:
for Model, name in [(SimpleCNN, "SimpleCNN"), (MediumCNN, "MediumCNN"), (DeepCNN, "DeepCNN")]:
    model = Model()
    out = model(x)
    loss = out.sum()
    loss.backward()
    grads_ok = all(p.grad is not None for p in model.parameters())
    print(f"{name}: gradients flowing = {grads_ok}")

SimpleCNN: gradients flowing = True
MediumCNN: gradients flowing = True
DeepCNN: gradients flowing = True


In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

for Model, name in [(SimpleCNN, "SimpleCNN"), (MediumCNN, "MediumCNN"), (DeepCNN, "DeepCNN")]:
    model = Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(100):
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
    acc = (out.argmax(1) == labels).float().mean().item()
    print(f"{name}: single batch acc = {acc:.2%}")

SimpleCNN: single batch acc = 100.00%
MediumCNN: single batch acc = 57.81%
DeepCNN: single batch acc = 25.00%


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

for Model, name, dropout in [(MediumCNN, "MediumCNN", None), (DeepCNN, "DeepCNN", 0.05)]:
    model = Model(dropout=dropout).to(device) if dropout else Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(100):
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
    acc = (out.argmax(1) == labels).float().mean().item()
    print(f"{name}: single batch acc = {acc:.2%}")

MediumCNN: single batch acc = 68.75%
DeepCNN: single batch acc = 81.25%


In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

for Model, name, dropout in [(MediumCNN, "MediumCNN", None), (DeepCNN, "DeepCNN", 0.01)]:
    model = Model(dropout=dropout).to(device) if dropout else Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(300):
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
    acc = (out.argmax(1) == labels).float().mean().item()
    print(f"{name}: single batch acc = {acc:.2%}")

MediumCNN: single batch acc = 78.12%
DeepCNN: single batch acc = 90.62%


In [32]:
for Model, name, dropout in [(MediumCNN, "MediumCNN", None), (DeepCNN, "DeepCNN", 0.01)]:
    model = Model(dropout=dropout).to(device) if dropout else Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(1000):
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
    acc = (out.argmax(1) == labels).float().mean().item()
    print(f"{name}: single batch acc = {acc:.2%}")

MediumCNN: single batch acc = 78.12%
DeepCNN: single batch acc = 95.31%


In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

model = MediumCNN().to(device)
for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.p = 0.0

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()
for epoch in range(500):
    optimizer.zero_grad()
    out = model(images)
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()
acc = (out.argmax(1) == labels).float().mean().item()
print(f"MediumCNN no dropout: single batch acc = {acc:.2%}")

MediumCNN no dropout: single batch acc = 100.00%


In [34]:
model = DeepCNN(dropout=0.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()
for epoch in range(500):
    optimizer.zero_grad()
    out = model(images)
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()
acc = (out.argmax(1) == labels).float().mean().item()
print(f"DeepCNN no dropout: single batch acc = {acc:.2%}")

DeepCNN no dropout: single batch acc = 100.00%


In [22]:
from src.models import SimpleCNN
from src.train import train
from src.dataset import get_dataloaders

DATA_PATH = "/root/.cache/kagglehub/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"

train_loader, val_loader = get_dataloaders(DATA_PATH, batch_size=64)

config = {
    "run_name": "simplecnn-baseline",
    "model": "SimpleCNN",
    "epochs": 20,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "Adam",
}

model = SimpleCNN()
train(model, train_loader, val_loader, config)

Epoch 1/20 | Train Loss: 1.6948 Acc: 0.3203 | Val Loss: 1.5818 Acc: 0.4003


Epoch 2/20 | Train Loss: 1.5294 Acc: 0.4130 | Val Loss: 1.4852 Acc: 0.4334


Epoch 3/20 | Train Loss: 1.4143 Acc: 0.4535 | Val Loss: 1.4095 Acc: 0.4571


Epoch 4/20 | Train Loss: 1.3323 Acc: 0.4878 | Val Loss: 1.3634 Acc: 0.4769


Epoch 5/20 | Train Loss: 1.2665 Acc: 0.5168 | Val Loss: 1.3466 Acc: 0.4879


...

Epoch 18/20 | Train Loss: 0.4900 Acc: 0.8268 | Val Loss: 1.8633 Acc: 0.5051


Epoch 19/20 | Train Loss: 0.4366 Acc: 0.8488 | Val Loss: 1.9141 Acc: 0.5078


Epoch 20/20 | Train Loss: 0.3799 Acc: 0.8685 | Val Loss: 2.0061 Acc: 0.5039


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇██
train/loss,█▇▇▆▆▅▅▅▄▄▄▄▃▃▂▂▂▂▁▁
val/accuracy,▁▃▄▆▆▇█▇█▇██▇▇▇▇▇▇▇▇
val/loss,▄▃▂▂▂▁▁▁▁▁▂▂▂▃▄▅▆▇▇█
epoch,20
train/accuracy,0.86847
train/loss,0.37993
val/accuracy,0.50392
val/loss,2.00614



Best Val Accuracy: 0.5187


In [ ]:
from src.models import MediumCNN

config = {
    "run_name": "mediumcnn-baseline",
    "model": "MediumCNN",
    "epochs": 20,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "Adam",
}

model = MediumCNN()
train(model, train_loader, val_loader, config)

Epoch 1/20 | Train Loss: 1.7209 Acc: 0.3112 | Val Loss: 1.4973 Acc: 0.4057


Epoch 2/20 | Train Loss: 1.4437 Acc: 0.4398 | Val Loss: 1.3941 Acc: 0.4646


Epoch 3/20 | Train Loss: 1.3257 Acc: 0.4891 | Val Loss: 1.3165 Acc: 0.4875


Epoch 4/20 | Train Loss: 1.2481 Acc: 0.5250 | Val Loss: 1.2442 Acc: 0.5196


Epoch 5/20 | Train Loss: 1.1862 Acc: 0.5469 | Val Loss: 1.2610 Acc: 0.5335


...

Epoch 18/20 | Train Loss: 0.3873 Acc: 0.8549 | Val Loss: 1.9012 Acc: 0.5870


Epoch 19/20 | Train Loss: 0.3558 Acc: 0.8696 | Val Loss: 2.0784 Acc: 0.5440


Epoch 20/20 | Train Loss: 0.3343 Acc: 0.8772 | Val Loss: 2.1302 Acc: 0.5710


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇███
train/loss,█▇▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁
val/accuracy,▁▃▄▅▆▆▇▆▇█▇▇█▇██▇█▆▇
val/loss,▃▃▂▂▂▁▁▂▂▁▂▂▃▄▄▄▆▆██
epoch,20
train/accuracy,0.87722
train/loss,0.33428
val/accuracy,0.57098
val/loss,2.13018



Best Val Accuracy: 0.5870


In [ ]:
from src.models import DeepCNN

config = {
    "run_name": "deepcnn-baseline",
    "model": "DeepCNN",
    "epochs": 20,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "Adam",
}

model = DeepCNN()
train(model, train_loader, val_loader, config)

epoch,▁▅█
train/accuracy,▁▆█
train/loss,█▃▁
val/accuracy,▁▆█
val/loss,█▃▁
epoch,3
train/accuracy,0.46478
train/loss,1.39753
val/accuracy,0.46002
val/loss,1.41836


Epoch 1/20 | Train Loss: 1.8541 Acc: 0.2420 | Val Loss: 1.7476 Acc: 0.2952


Epoch 2/20 | Train Loss: 1.7372 Acc: 0.2888 | Val Loss: 1.6267 Acc: 0.3160


Epoch 3/20 | Train Loss: 1.6043 Acc: 0.3526 | Val Loss: 1.5608 Acc: 0.4093


Epoch 4/20 | Train Loss: 1.5394 Acc: 0.3814 | Val Loss: 1.5058 Acc: 0.4329


...

Epoch 18/20 | Train Loss: 1.1741 Acc: 0.5406 | Val Loss: 1.2104 Acc: 0.5506


Epoch 19/20 | Train Loss: 1.1416 Acc: 0.5584 | Val Loss: 1.2188 Acc: 0.5633


Epoch 20/20 | Train Loss: 1.1095 Acc: 0.5711 | Val Loss: 1.2027 Acc: 0.5656


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▂▃▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇██
train/loss,█▇▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁
val/accuracy,▁▂▄▅▅▅▅▅▅▅▅▅▆▆▆▇▇███
val/loss,█▆▆▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
epoch,20
train/accuracy,0.5711
train/loss,1.10946
val/accuracy,0.56558
val/loss,1.2027



Best Val Accuracy: 0.5656


In [ ]:
!cat /content/facial-expression-recognition/src/models.py


import torch
import torch.nn as nn

# Model 1: Simple CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Model 2: Medium CNN
class MediumCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(MediumCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.Re

In [24]:
import importlib
import src.models
importlib.reload(src.models)
from src.models import SimpleCNN, MediumCNN, DeepCNN

In [25]:
model = DeepCNN(dropout=0.1)
print("OK")

OK


In [ ]:
config = {
    "run_name": "deepcnn-low-dropout",
    "model": "DeepCNN",
    "epochs": 20,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "Adam",
    "dropout": 0.1,
}
model = DeepCNN(dropout=0.1)
train(model, train_loader, val_loader, config)

Epoch 1/20 | Train Loss: 1.8120 Acc: 0.2601 | Val Loss: 1.7436 Acc: 0.2972


Epoch 2/20 | Train Loss: 1.6009 Acc: 0.3617 | Val Loss: 1.4758 Acc: 0.4276


Epoch 3/20 | Train Loss: 1.4356 Acc: 0.4394 | Val Loss: 1.3735 Acc: 0.4637


Epoch 4/20 | Train Loss: 1.3313 Acc: 0.4798 | Val Loss: 1.2796 Acc: 0.5022


Epoch 5/20 | Train Loss: 1.2635 Acc: 0.5102 | Val Loss: 1.2352 Acc: 0.5133


...

Epoch 18/20 | Train Loss: 0.6065 Acc: 0.7709 | Val Loss: 1.1670 Acc: 0.6098


Epoch 19/20 | Train Loss: 0.5600 Acc: 0.7925 | Val Loss: 1.1970 Acc: 0.6069


Epoch 20/20 | Train Loss: 0.5006 Acc: 0.8115 | Val Loss: 1.2691 Acc: 0.5997


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▂▃▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██
train/loss,█▇▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁
val/accuracy,▁▄▅▅▆▆▇▇▇▇▇▇██▇█████
val/loss,█▅▄▃▃▂▂▂▁▂▂▁▁▁▂▂▂▂▂▃
epoch,20
train/accuracy,0.81152
train/loss,0.50063
val/accuracy,0.59972
val/loss,1.26906



Best Val Accuracy: 0.6163


In [ ]:
config = {
    "run_name": "deepcnn-dropout015",
    "model": "DeepCNN",
    "epochs": 20,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "Adam",
    "dropout": 0.15,
}

model = DeepCNN(dropout=0.15)
train(model, train_loader, val_loader, config)

Epoch 1/20 | Train Loss: 1.8449 Acc: 0.2462 | Val Loss: 1.7678 Acc: 0.2759


Epoch 2/20 | Train Loss: 1.6650 Acc: 0.3340 | Val Loss: 1.5319 Acc: 0.4093


Epoch 3/20 | Train Loss: 1.4928 Acc: 0.4159 | Val Loss: 1.4136 Acc: 0.4639


Epoch 4/20 | Train Loss: 1.4018 Acc: 0.4459 | Val Loss: 1.4042 Acc: 0.4585


Epoch 5/20 | Train Loss: 1.3405 Acc: 0.4717 | Val Loss: 1.3304 Acc: 0.5008


...

Epoch 18/20 | Train Loss: 0.8936 Acc: 0.6596 | Val Loss: 1.1090 Acc: 0.5877


Epoch 19/20 | Train Loss: 0.8650 Acc: 0.6722 | Val Loss: 1.0852 Acc: 0.5954


Epoch 20/20 | Train Loss: 0.8217 Acc: 0.6887 | Val Loss: 1.1095 Acc: 0.5896


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▂▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
train/loss,█▇▆▅▅▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁
val/accuracy,▁▄▅▅▆▆▆▆▇▇▇▇████████
val/loss,█▆▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
epoch,20
train/accuracy,0.6887
train/loss,0.82165
val/accuracy,0.58962
val/loss,1.10949



Best Val Accuracy: 0.5961


In [ ]:
config = {
    "run_name": "mediumcnn-lr0001",
    "model": "MediumCNN",
    "epochs": 20,
    "lr": 0.0001,
    "batch_size": 64,
    "optimizer": "Adam",
}

model = MediumCNN()
train(model, train_loader, val_loader, config)

Epoch 1/20 | Train Loss: 1.6446 Acc: 0.3532 | Val Loss: 1.4830 Acc: 0.4398


Epoch 2/20 | Train Loss: 1.4049 Acc: 0.4639 | Val Loss: 1.3686 Acc: 0.4748


Epoch 3/20 | Train Loss: 1.2818 Acc: 0.5152 | Val Loss: 1.2978 Acc: 0.5132


Epoch 4/20 | Train Loss: 1.1801 Acc: 0.5585 | Val Loss: 1.3205 Acc: 0.4999


Epoch 5/20 | Train Loss: 1.0888 Acc: 0.5963 | Val Loss: 1.2606 Acc: 0.5227


...

Epoch 18/20 | Train Loss: 0.1211 Acc: 0.9681 | Val Loss: 1.9873 Acc: 0.5391


Epoch 19/20 | Train Loss: 0.1015 Acc: 0.9754 | Val Loss: 2.1082 Acc: 0.5415


Epoch 20/20 | Train Loss: 0.0870 Acc: 0.9796 | Val Loss: 2.0727 Acc: 0.5393


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▂▃▃▄▄▅▅▅▆▆▇▇▇██████
train/loss,█▇▆▆▆▅▅▄▄▃▃▂▂▂▂▁▁▁▁▁
val/accuracy,▁▃▆▅▆▇██▇▇█▇▇█▇█▇▇▇▇
val/loss,▃▂▂▂▁▁▁▁▂▂▂▃▅▄▅▅█▇██
epoch,20
train/accuracy,0.97962
train/loss,0.08698
val/accuracy,0.53928
val/loss,2.07273



Best Val Accuracy: 0.5503


In [23]:
config = {
    "run_name": "simplecnn-lr0001",
    "model": "SimpleCNN",
    "epochs": 20,
    "lr": 0.0001,
    "batch_size": 64,
    "optimizer": "Adam",
}

model = SimpleCNN()
train(model, train_loader, val_loader, config)

Epoch 1/20 | Train Loss: 1.7826 Acc: 0.2611 | Val Loss: 1.7416 Acc: 0.2846


Epoch 2/20 | Train Loss: 1.6996 Acc: 0.3166 | Val Loss: 1.6664 Acc: 0.3423


Epoch 3/20 | Train Loss: 1.6360 Acc: 0.3639 | Val Loss: 1.6182 Acc: 0.3869


Epoch 4/20 | Train Loss: 1.6004 Acc: 0.3850 | Val Loss: 1.5883 Acc: 0.3926


Epoch 5/20 | Train Loss: 1.5774 Acc: 0.3952 | Val Loss: 1.5953 Acc: 0.3794


...

Epoch 18/20 | Train Loss: 1.4072 Acc: 0.4663 | Val Loss: 1.4451 Acc: 0.4485


Epoch 19/20 | Train Loss: 1.3956 Acc: 0.4688 | Val Loss: 1.4276 Acc: 0.4553


Epoch 20/20 | Train Loss: 1.3835 Acc: 0.4760 | Val Loss: 1.4321 Acc: 0.4525


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/accuracy,▁▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/loss,█▇▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
val/accuracy,▁▃▅▅▅▆▆▆▇▇▇▇▇▇▇█████
val/loss,█▆▅▅▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁
epoch,20
train/accuracy,0.47597
train/loss,1.38352
val/accuracy,0.45253
val/loss,1.43214



Best Val Accuracy: 0.4553


In [24]:
%%writefile /content/facial-expression-recognition/src/train.py

import torch
import torch.nn as nn
import wandb
from tqdm import tqdm

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

def train(model, train_loader, val_loader, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if config["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    elif config["optimizer"] == "AdamW":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config["lr"],
            weight_decay=config.get("weight_decay", 0.01)
        )
    elif config["optimizer"] == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"], momentum=0.9)

    wandb.init(
        project="facial-expression-recognition",
        name=config["run_name"],
        config=config
    )
    wandb.watch(model, log="all")

    best_val_acc = 0

    for epoch in range(config["epochs"]):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = val_epoch(model, val_loader, criterion, device)

        wandb.log({
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
        })

        print(f"Epoch {epoch+1}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{config['run_name']}_best.pth")

    wandb.finish()
    print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Overwriting /content/facial-expression-recognition/src/train.py


In [25]:
config = {
    "run_name": "deepcnn-adamw-dropout010",
    "model": "DeepCNN",
    "epochs": 30,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "AdamW",
    "dropout": 0.10,
    "weight_decay": 0.01,
}

model = DeepCNN(dropout=0.10)
train(model, train_loader, val_loader, config)

Epoch 1/30 | Train Loss: 1.8435 Acc: 0.2439 | Val Loss: 1.7978 Acc: 0.2419


Epoch 2/30 | Train Loss: 1.6645 Acc: 0.3271 | Val Loss: 1.5576 Acc: 0.3816


Epoch 3/30 | Train Loss: 1.4724 Acc: 0.4205 | Val Loss: 1.3838 Acc: 0.4689


Epoch 4/30 | Train Loss: 1.3522 Acc: 0.4707 | Val Loss: 1.3234 Acc: 0.4902


Epoch 5/30 | Train Loss: 1.2768 Acc: 0.5024 | Val Loss: 1.2561 Acc: 0.5201


...

Epoch 28/30 | Train Loss: 0.2821 Acc: 0.8979 | Val Loss: 1.3440 Acc: 0.6163


Epoch 29/30 | Train Loss: 0.2614 Acc: 0.9037 | Val Loss: 1.4476 Acc: 0.6323


Epoch 30/30 | Train Loss: 0.2468 Acc: 0.9100 | Val Loss: 1.4096 Acc: 0.6283


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/accuracy,▁▂▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇██████
train/loss,█▇▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
val/accuracy,▁▄▅▅▆▆▇▇▇▇▇█████▇█████████████
val/loss,█▆▄▄▃▃▂▂▂▁▁▁▁▁▁▁▂▁▂▂▂▃▃▃▄▃▄▄▅▄
epoch,30
train/accuracy,0.91005
train/loss,0.24684
val/accuracy,0.62829
val/loss,1.40965



Best Val Accuracy: 0.6323


In [26]:
config = {
    "run_name": "deepcnn-adamw-dropout012",
    "model": "DeepCNN",
    "epochs": 30,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "AdamW",
    "dropout": 0.12,
    "weight_decay": 0.01,
}

model = DeepCNN(dropout=0.12)
train(model, train_loader, val_loader, config)

Epoch 1/30 | Train Loss: 1.8509 Acc: 0.2489 | Val Loss: 1.7568 Acc: 0.2761


Epoch 2/30 | Train Loss: 1.6593 Acc: 0.3345 | Val Loss: 1.5310 Acc: 0.4078


Epoch 3/30 | Train Loss: 1.4698 Acc: 0.4266 | Val Loss: 1.4073 Acc: 0.4438


Epoch 4/30 | Train Loss: 1.3609 Acc: 0.4703 | Val Loss: 1.2850 Acc: 0.5138


Epoch 5/30 | Train Loss: 1.2884 Acc: 0.4987 | Val Loss: 1.2481 Acc: 0.5154


...

Epoch 28/30 | Train Loss: 0.3285 Acc: 0.8826 | Val Loss: 1.3453 Acc: 0.6276


Epoch 29/30 | Train Loss: 0.3136 Acc: 0.8864 | Val Loss: 1.3198 Acc: 0.6173


Epoch 30/30 | Train Loss: 0.2887 Acc: 0.8959 | Val Loss: 1.3313 Acc: 0.6241


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/accuracy,▁▂▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇██████
train/loss,█▇▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
val/accuracy,▁▄▄▆▆▆▆▆▇▇▇▇▇██▇██████████████
val/loss,█▆▅▃▃▂▂▂▂▂▂▂▁▁▁▁▂▁▁▂▂▂▂▃▃▃▃▄▄▄
epoch,30
train/accuracy,0.89586
train/loss,0.2887
val/accuracy,0.62411
val/loss,1.33126



Best Val Accuracy: 0.6276


In [35]:
!cd /content/facial-expression-recognition && \
  git config user.email "tgela23@freeuni.edu.ge" && \
  git config user.name "tamari1990" && \
  git add . && \
  git commit -m "Add src files and notebook" && \
  git push

[main 2137de5] Add src files and notebook
 28 files changed, 3730 insertions(+)
 create mode 100644 deepcnn-adamw-dropout010_best.pth
 create mode 100644 deepcnn-adamw-dropout012_best.pth
 create mode 100644 simplecnn-baseline_best.pth
 create mode 100644 simplecnn-lr0001_best.pth
 create mode 100644 src/dataset.py
 create mode 100644 src/models.py
 create mode 100644 src/train.py
 create mode 120000 wandb/latest-run
 create mode 100644 wandb/run-20260615_141429-v0vyutb9/files/config.yaml
 create mode 100644 wandb/run-20260615_141429-v0vyutb9/files/requirements.txt
 create mode 100644 wandb/run-20260615_141429-v0vyutb9/files/wandb-metadata.json
 create mode 100644 wandb/run-20260615_141429-v0vyutb9/files/wandb-summary.json
 create mode 100644 wandb/run-20260615_141429-v0vyutb9/run-v0vyutb9.wandb
 create mode 100644 wandb/run-20260615_142039-9ijmqzzj/files/config.yaml
 create mode 100644 wandb/run-20260615_142039-9ijmqzzj/files/requirements.txt
 create mode 100644 wandb/run-20260615_142

In [38]:
!git clone https://github.com/tamari1990/facial-expression-recognition.git

Cloning into 'facial-expression-recognition'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [39]:
!mkdir -p facial-expression-recognition/src
!cp -r src/* facial-expression-recognition/src/
!cp facial_expression.ipynb facial-expression-recognition/

cp: cannot stat 'facial_expression.ipynb': No such file or directory


In [40]:
%cd /content/facial-expression-recognition

!git status

/content/facial-expression-recognition
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	facial-expression-recognition/

nothing added to commit but untracked files present (use "git add" to track)


In [41]:
!rm -rf facial-expression-recognition/

In [42]:
!git status
!git push

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
remote: Permission to tamari1990/facial-expression-recognition.git denied to tamari1990.
fatal: unable to access 'https://github.com/tamari1990/facial-expression-recognition.git/': The requested URL returned error: 403


In [43]:
!git push

remote: Permission to tamari1990/facial-expression-recognition.git denied to tamari1990.
fatal: unable to access 'https://github.com/tamari1990/facial-expression-recognition.git/': The requested URL returned error: 403


In [46]:
!find /content/drive/MyDrive -name "*.ipynb"

find: ‘/content/drive/MyDrive’: No such file or directory


In [47]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [48]:
!find "/content/drive/MyDrive" -name "*.ipynb"

/content/drive/MyDrive/Colab Notebooks/facial_expression.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Classroom/Natural Language Processing (Spring 2026)/tgela23 (2).ipynb
/content/drive/MyDrive/Untitled0.ipynb
/content/drive/MyDrive/cs231n/assignments/assignment2/BatchNormalization.ipynb
/content/drive/MyDrive/cs231n/assignments/assignment2/Dropout.ipynb
/content/drive/MyDrive/cs231n/assignments/assignment2/FullyConnectedNets.ipynb
/content/drive/MyDrive/cs231n/assignments/assignment2/PyTorch.ipynb
/content/drive/MyDrive/cs231n/assignments/assignment2/ConvolutionalNetworks.ipynb
/content/drive/MyDrive/cs231n/assignments/assignment2/collect_submission.ipynb


In [51]:
!cp "/content/drive/MyDrive/Colab Notebooks/facial_expression.ipynb" /content/facial-expression-recognition/

%cd /content/facial-expression-recognition

!git add facial_expression.ipynb
!git commit -m "add notebook"
!git push

/content/facial-expression-recognition
[main f95a063] add notebook
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 19.14 KiB | 6.38 MiB/s, done.
Total 9 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 1 local object.
remote: error: GH013: Repository rule violations found for refs/heads/main.
remote: 
remote: - GITHUB PUSH PROTECTION
remote:   —————————————————————————————————————————
remote:     Resolve the following violations before pushing again
remote: 
remote:     - Push cannot contain secrets
remote: 
remote:     
remote:      (?) Learn how to resolve a blocked push
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push
rem

In [52]:
!grep -R "ghp_" .
!grep -R "github_pat_" .

./facial_expression.ipynb:{"nbformat":4,"nbformat_minor":0,"metadata":{"colab":{"provenance":[],"gpuType":"T4","authorship_tag":"ABX9TyNFQkdV0pJEMaSkHMvy43Yj"},"kernelspec":{"name":"python3","display_name":"Python 3"},"language_info":{"name":"python"},"accelerator":"GPU"},"cells":[{"cell_type":"code","execution_count":1,"metadata":{"colab":{"base_uri":"https://localhost:8080/"},"id":"mP3gyPNOo50y","executionInfo":{"status":"ok","timestamp":1781532729807,"user_tz":420,"elapsed":748,"user":{"displayName":"Tamari Gelashvili","userId":"15675067474287235040"}},"outputId":"816ced01-c2d0-4fad-dbfa-b80de6464224"},"outputs":[{"output_type":"stream","name":"stdout","text":["Cloning into 'facial-expression-recognition'...\n","remote: Enumerating objects: 4, done.\u001b[K\n","remote: Counting objects: 100% (4/4), done.\u001b[K\n","remote: Compressing objects: 100% (3/3), done.\u001b[K\n","remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)\u001b[K\n","Receiving objects: 100% (4/4